In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import requests
import io
import googlemaps
from pathlib import Path

### Data Gathering:

The base data gathering will consist of creating one main hospital-level DataFrame with one row per hospital containing identifying information such as Medicare CCN number and NPI number. This will serve as a base for the project; yearly financial data, hospital quality data, etc. will get merged in later.

In [2]:
cms_providers = pd.read_csv('../data/Hospital_General_Information.csv')

In [3]:
cms_providers.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29.0,15,10,29.0,10,7,29.0


In [4]:
cms_providers.columns

Index(['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State',
       'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type',
       'Hospital Ownership', 'Emergency Services',
       'Meets criteria for birthing friendly designation',
       'Hospital overall rating', 'Hospital overall rating footnote',
       'MORT Group Measure Count', 'Count of Facility MORT Measures',
       'Count of MORT Measures Better', 'Count of MORT Measures No Different',
       'Count of MORT Measures Worse', 'MORT Group Footnote',
       'Safety Group Measure Count', 'Count of Facility Safety Measures',
       'Count of Safety Measures Better',
       'Count of Safety Measures No Different',
       'Count of Safety Measures Worse', 'Safety Group Footnote',
       'READM Group Measure Count', 'Count of Facility READM Measures',
       'Count of READM Measures Better',
       'Count of READM Measures No Different', 'Count of READM Measures Worse',
       'READM Group Footnote', 'Pt Exp Gr

In [5]:
cms_providers['Hospital Type'].unique()

array(['Acute Care Hospitals', 'Acute Care - Veterans Administration',
       'Rural Emergency Hospital', 'Critical Access Hospitals',
       'Childrens', 'Psychiatric', 'Acute Care - Department of Defense',
       'Long-term'], dtype=object)

Narrowing down to just Acute Care Hospitals:

In [6]:
cms_providers = cms_providers[cms_providers['Hospital Type'] == 'Acute Care Hospitals']

Selecting only the columns that we need:

In [7]:
cms_providers = cms_providers[['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Emergency Services','Hospital Ownership']]

Renaming Facility ID to CCN for clarity:

In [8]:
cms_providers = cms_providers.rename(columns={'Facility ID': 'CCN', 'County/Parish': 'County', 'City/Town': 'City', 'ZIP Code': 'Zip'})


Adding a Closure column and setting to 0 since these hospitals are currently open.

In [9]:
 cms_providers['Closed'] = 0

In [10]:
cms_providers.head(2)

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,Yes,Government - Hospital District or Authority,0
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,Yes,Government - Hospital District or Authority,0


Reading in rural hospital closures data:

In [11]:
rural_hospital_closures = pd.read_excel('../data/Closures-Database-for-Web.xlsx')

In [12]:
rural_hospital_closures.head()

,Hospital,Address,City,State,Zip,County/district,RUCA,CBSA,Medicare Payment,# of Beds,Closure Month,Closure Year,Complete Closure (0);\nConverted Closure (1),updated 5/18/2025
0,Bradford Regional Medical Center,116 INTERSTATE PARKWAY,BRADFORD,PA,16701.0,McKean,4.0,Micro,PPS,14.0,May,2026.0,1,196 Closed Rural Hospitals 2005-2026
1,Claremore Indian Hospital,101 South Moore Ave,Claremore,OK,74017.0,Rogers,4.0,Metro,IHS,46.0,October,2025.0,1,88 Converted to other health services
2,Glenn Medical Center,1133 W SYCAMORE ST,WILLOWS,CA,95988.0,Glenn,7.0,Neither,CAH,25.0,September,2025.0,1,NaN
3,Northern Light Inland Hospital,200 KENNEDY MEMORIAL DRIVE,WATERVILLE,ME,4901.0,Kennebec,4.0,Micro,MDH,33.0,May,2025.0,0,NaN
4,Lawrence Medical Center,202 HOSPITAL STREET,MOULTON,AL,35650.0,Lawrence,7.0,Metro,PPS,43.0,May,2025.0,1,NaN


In [13]:
rural_hospital_closures.columns

Index(['Hospital', 'Address', 'City', 'State', 'Zip', 'County/district',
       'RUCA', 'CBSA', 'Medicare Payment', '# of Beds', 'Closure Month',
       'Closure Year', 'Complete Closure (0);\nConverted Closure (1)',
       'updated 5/18/2025'],
      dtype='object')

Updating closure column so that complete closure = 2 and converted closure = 1:

In [14]:
rural_hospital_closures = rural_hospital_closures.rename(columns={'Complete Closure (0);\nConverted Closure (1)': 'Closed', 'County/district':'County', '# of Beds':'Number of Beds','Hospital':'Facility Name'})


In [15]:
rural_hospital_closures['Closed'] = rural_hospital_closures['Closed'].replace(0,2)

Remove footer:

In [16]:
rural_hospital_closures = rural_hospital_closures[rural_hospital_closures['Closed'] != '88 Converted to other health services']

Narrow down to just the columns we need and drop any fully null rows:

In [17]:
rural_hospital_closures = (rural_hospital_closures.drop(columns=['updated 5/18/2025','CBSA']).dropna(how='all'))

Cast ZIP, number of beds, and closure year to integer:

In [18]:
rural_hospital_closures[['Zip','Number of Beds','Closure Year']] = rural_hospital_closures[['Zip','Number of Beds','Closure Year']].astype(int)


Convert closure month/year to date:

In [19]:
rural_hospital_closures['Closure Month'] = pd.to_datetime(rural_hospital_closures['Closure Month'], format='%B', errors='coerce').dt.month

In [20]:
rural_hospital_closures = rural_hospital_closures.rename(columns={'Closure Year':'year', 'Closure Month':'month'})

In [21]:
rural_hospital_closures['Closure Date'] = pd.to_datetime(rural_hospital_closures[['year', 'month']].assign(day=1))

In [22]:
rural_hospital_closures = rural_hospital_closures.drop(columns=['year','month'])

In [23]:
rural_hospital_closures.head(2)

,Facility Name,Address,City,State,Zip,County,RUCA,Medicare Payment,Number of Beds,Closed,Closure Date
0,Bradford Regional Medical Center,116 INTERSTATE PARKWAY,BRADFORD,PA,16701,McKean,4.0,PPS,14,1,2026-05-01
1,Claremore Indian Hospital,101 South Moore Ave,Claremore,OK,74017,Rogers,4.0,IHS,46,1,2025-10-01


In [176]:
rural_hospital_closures[rural_hospital_closures['RUCA']==1.0]

,Facility Name,Address,City,State,Zip,County,RUCA,Medicare Payment,Number of Beds,Closed,Closure Date
117,Mid-Valley Hospital,1400 MAIN STREET,Peckville,PA,18452,Lackawanna,1.0,CAH,25,1,2014-06-01
172,Ft Yuma PHS Indian Hosp,PO BOX 1368,Yuma,AZ,85366,Yuma,1.0,CAH,15,2,2007-06-01


In [24]:
rural_hospital_closures['Closed'].value_counts()

Closed
2    108
1     88
Name: count, dtype: int64

Combine currently active hospital data with closed hospital data:

In [25]:
hospitals_master = pd.concat([cms_providers, rural_hospital_closures]).reset_index()

Uppercase all hospital names and addresses for consistency:

In [26]:
hospitals_master[['Facility Name','Address', 'City', 'State', 'County']] = (
                                                                                     hospitals_master[['Facility Name','Address', 'City', 'State', 'County']]
                                                                                     .apply(lambda x : x.str.upper())
                                                                                    )

Pre-process addresses to help with geocoding:

In [27]:
# Keep only the first comma-delimited segment (Some addresses already contain the city and state; we'll strip that off.)
hospitals_master['Address'] = hospitals_master['Address'].str.split(",").str[0]

# Remove suite / box / mail-slot style text
hospitals_master['Address'] = (
    hospitals_master['Address']
    .str.replace(
        r"\s*(?:SUITE|UNIT|STE|PO BOX|BOX|P O BOX|POST OFFICE BOX|MAIL SLOT)\s*\.?\s*\d+",
        "",
        case=False,
        regex=True,
    )
    .str.strip()
)

# Expand common written numbers / ordinals
mapping = {
    "FIRST ": "1ST ",
    "SECOND ": "2ND ",
    "THIRD ": "3RD ",
    "FOURTH ": "4TH ",
    "FIFTH ": "5TH ",
    "SIXTH ": "6TH ",
    "SEVENTH ": "7TH ",
    "EIGHTH ": "8TH ",
    "NINTH ": "9TH ",
    "TENTH ": "10TH ",
    "ONE ": "1 ",
    "TWO ": "2 ",
    "THREE ": "3 ",
    "FOUR ": "4 ",
    "FIVE ": "5 ",
    "SIX ": "6 ",
    "SEVEN ": "7 ",
    "EIGHT ": "8 ",
    "NINE ": "9 ",
    "TEN ": "10 ",
    "TWNSHP PRKWY": "TOWNSHIP PARKWAY",
}

for word, replacement in mapping.items():
    hospitals_master['Address'] = hospitals_master['Address'].str.replace(word, replacement, regex=False)

Combine address parts to make a full address column.

In [28]:
hospitals_master['Full Address'] = (
        hospitals_master['Address'] + ", " +
        hospitals_master['City'] + ", " +
        hospitals_master['State'] + ", " +
        hospitals_master['Zip'].astype(str)
    )

In [29]:
hospitals_master

,index,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,RUCA,Medicare Payment,Number of Beds,Closure Date,Full Address
0,0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaN,NaT,"1108 ROSS CLARK CIRCLE, DOTHAN, AL, 36301"
1,1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaN,NaT,"2505 U S HIGHWAY 431 NORTH, BOAZ, AL, 35957"
2,2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,Yes,Proprietary,0,NaN,NaN,NaN,NaT,"1701 VETERANS DRIVE, FLORENCE, AL, 35630"
3,3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaN,NaT,"702 N MAIN ST, OPP, AL, 36467"
4,4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaN,NaT,"50 MEDICAL PARK EAST DRIVE, BIRMINGHAM, AL, 35235"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3306,191,NaN,FAYETTE MEMORIAL HOSPITAL,543 N JACKSON,LA GRANGE,TX,78945,FAYETTE,NaN,NaN,2,7.0,PPS,24.0,2005-07-01,"543 N JACKSON, LA GRANGE, TX, 78945"
3307,192,NaN,DELEON HOSPITAL,407 S TEXAS,DE LEON,TX,76444,COMANCHE,NaN,NaN,2,10.0,CAH,14.0,2005-07-01,"407 S TEXAS, DE LEON, TX, 76444"
3308,193,NaN,MINNEWASKA DISTRICT HOSP,610 W 6TH ST,STARBUCK,MN,56381,POPE,NaN,NaN,2,10.0,CAH,19.0,2005-06-01,"610 W 6TH ST, STARBUCK, MN, 56381"
3309,194,NaN,GULF PINES HOSPITAL,102 20TH ST,PORT SAINT JOE,FL,32456,GULF,NaN,NaN,2,7.1,PPS,45.0,2005-03-01,"102 20TH ST, PORT SAINT JOE, FL, 32456"


In [30]:
hospitals_master.shape

(3311, 16)

Now we'll use the US Census Bureau's geocoder tool to pull in FIPS codes as well as census tract numbers.

In [31]:
addresses = (hospitals_master
             [['Address','City','State','Zip']]
             .rename(columns={'Address':'Street Address','Zip':'Zip Code'})
            )

addresses.to_csv('../data/addresses.csv', header=False)

In [32]:
addresses.shape

(3311, 4)

In [33]:
# Define endpoint and parameters
url = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
payload = {
    'benchmark': 'Public_AR_Current',
    'vintage': 'Current_Current' 
}

# Open csv file and send post request
file_path = "../data/addresses.csv"
files = {'addressFile': (file_path, open(file_path, 'rb'), 'text/csv')}

response = requests.post(url, files=files, data=payload)

# Parse the output into a DataFrame
# The census API returns a csv-like text string. We name the expected columns based on the API documentation.
column_names = [
    'Unique_ID', 'Input_Address', 'Match_Status', 'Match_Type', 'Matched_Address', 
    'Coordinates', 'Tiger_Line_ID', 'Tiger_Side', 'State_FIPS', 'County_FIPS', 
    'Tract_Code', 'Block_Code'
]

if response.status_code == 200:
    res_df = pd.read_csv(io.StringIO(response.text), header=None, names=column_names)[['Input_Address','Coordinates','Tract_Code']]
    hospitals_master_geo = hospitals_master.merge(res_df, left_on='Full Address', right_on='Input_Address', how='left')
else:
    print(f"Error: {response.status_code}")

I also geocoded with Nominatim which resulted in a few more facilities able to be successfully geocoded. Reading in that data:

In [34]:
nom = pd.read_csv('../data/nominatim_geocoded.csv')

In [35]:
hospitals_master_geo_full = hospitals_master_geo.merge(nom, on=['Facility Name','Address'], how='left')

In [36]:
# Coalesce the coordinates column using combine_first
hospitals_master_geo_full['Coordinates'] = hospitals_master_geo_full['Coordinates_x'].combine_first(hospitals_master_geo_full['Coordinates_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['Coordinates_x', 'Coordinates_y', 'Input_Address','index']).drop_duplicates()

In [37]:
# Cast any 'nan, nan' coordinates to a true nan
hospitals_master_geo_full['Coordinates'] = hospitals_master_geo_full['Coordinates'].replace('nan, nan', np.nan)

In [38]:
hospitals_master_geo_full['Coordinates'].value_counts()

Coordinates
-95.379539091063,30.954872977908    2
-82.79316256635,36.173564570249     2
-117.261201132,34.049923287677      2
41.32221, -96.01991                 2
-97.21409421228,34.720187305292     2
                                   ..
37.3202977, -95.2651809             1
-97.331735804997,37.700183290544    1
-97.29897357354,37.694583872852     1
-95.247675465623,38.979415033937    1
-91.051497672236,33.404693930931    1
Name: count, Length: 3221, dtype: int64

We see some coordinates are repeated; some duplicates should be dropped, and some represent hospital name changes.

In [39]:
hospitals_master_geo_full[hospitals_master_geo_full.duplicated(subset=['Coordinates'], keep=False)][['Facility Name','Full Address','Coordinates']].sort_values(by='Coordinates').head(20)


,Facility Name,Full Address,Coordinates
328,LOMA LINDA UNIVERSITY MEDICAL CENTER,"11234 ANDERSON ST, LOMA LINDA, CA, 92354","-117.261201132,34.049923287677"
458,LOMA LINDA UNIVERSITY CHILDREN'S HOSPITAL,"11234 ANDERSON STREET SUITE A, LOMA LINDA, CA,...","-117.261201132,34.049923287677"
2525,GREENEVILLE COMMUNITY HOSPITAL,"1420 TUSCULUM BOULEVARD, GREENEVILLE, TN, 37745","-82.79316256635,36.173564570249"
3190,TAKOMA REGIONAL HOSPITAL,"1420 TUSCULUM BOULEVARD, GREENEVILLE, TN, 37745","-82.79316256635,36.173564570249"
2503,UNITY MEDICAL CENTER,"481 INTERSTATE DRIVE, MANCHESTER, TN, 37355","-86.077944643285,35.497800440307"
3237,UNITED REGIONAL MEDICAL CENTER,"481 INTERSTATE DRIVE, MANCHESTER, TN, 37355","-86.077944643285,35.497800440307"
1473,MAYO CLINIC HEALTH SYSTEM - ALBERT LEA AND AUSTIN,"404 WEST FOUNTAIN STREET, ALBERT LEA, MN, 56007","-93.372490760223,43.651590633716"
3185,ALBERT LEA - MAYO CLINIC HEALTH SYSTEM,"404 WEST FOUNTAIN STREET, ALBERT LEA, MN, 56007","-93.372490760223,43.651590633716"
3117,MID COAST MEDICAL CENTER-TRINITY,"317 PROSPECT DRIVE, TRINITY, TX, 75862","-95.379539091063,30.954872977908"
3135,MID COAST MEDICAL CENTER - TRINITY,"317 PROSPECT DRIVE, TRINITY, TX, 75862","-95.379539091063,30.954872977908"


In [40]:
hospitals_master_geo_full['Prior Name'] = ''

In [41]:
hospitals_master_geo_full.loc[2525, 'Prior Name'] = 'TAKOMA REGIONAL HOSPITAL'
hospitals_master_geo_full.loc[2503, 'Prior Name'] = 'UNITED REGIONAL MEDICAL CENTER'

In [42]:
hospitals_master_geo_full = hospitals_master_geo_full.drop([3190,3237,1473,3117,2166,2196,1662])

Filling in the final few facilities with missing coordinates:

In [43]:
# Reading in Google Maps api key
with open('./keys.json') as fi:
    credentials = json.load(fi)

api_key = credentials['api_key']

In [44]:
missing_coords = hospitals_master_geo_full[hospitals_master_geo_full['Coordinates'].isna()][['Facility Name', 'Full Address']]

In [45]:
missing_coords

,Facility Name,Full Address
31,RUSSELL MEDICAL CENTER,"3316 HIGHWAY 280, ALEXANDER CITY, AL, 35010"
32,CLAY COUNTY HOSPITAL,"83825 HIGHWAY 9, ASHLAND, AL, 36251"
52,WHITFIELD REGIONAL HOSPITAL,"105 HIGHWAY 80 EAST, DEMOPOLIS, AL, 36732"
57,LAKELAND COMMUNITY HOSPITAL,"42024 HIGHWAY 195 E, HALEYVILLE, AL, 35565"
70,RUSSELLVILLE HOSPITAL,"15155 HIGHWAY 43, RUSSELLVILLE, AL, 35653"
...,...,...
3295,JENKINS COMMUNITY HOSPITAL,"9480 HIGHWAY 805, JENKINS, KY, 41537"
3302,RICHWOOD AREA COMMUNITY HOSPITAL,"RIVERSIDE ADDITION, RICHWOOD, WV, 26261"
3306,OWYHEE COMMUNITY HEALTH FACILITY,", OWYHEE, NV, 89832"
3309,FT YUMA PHS INDIAN HOSP,", YUMA, AZ, 85366"


In [46]:
gmaps = googlemaps.Client(key=api_key)

def gmaps_geocode(row):
    """
    Geocodes an address string (falling back to facility name if needed) and returns latitude and longitude series.
    """
    address = row['Full Address']
    name = row['Facility Name']

    geocode_result = gmaps.geocode(address)
    if geocode_result:
        location = geocode_result[0]["geometry"]["location"]
        return pd.Series([location["lat"], location["lng"]], index=["Latitude", "Longitude"])
    else:
        geocode_result = gmaps.geocode(name)
        if geocode_result:
            location = geocode_result[0]["geometry"]["location"]
            return pd.Series([location["lat"], location["lng"]], index=["Latitude", "Longitude"])
        
    return pd.Series([None, None], index=["Latitude", "Longitude"])

missing_coords[["Latitude", "Longitude"]] = missing_coords.apply(gmaps_geocode, axis=1)

In [47]:
missing_coords['Coordinates'] = missing_coords['Latitude'].astype(str) + ', ' + missing_coords['Longitude'].astype(str)

In [48]:
missing_coords

,Facility Name,Full Address,Latitude,Longitude,Coordinates
31,RUSSELL MEDICAL CENTER,"3316 HIGHWAY 280, ALEXANDER CITY, AL, 35010",32.930053,-85.969726,"32.9300534, -85.96972629999999"
32,CLAY COUNTY HOSPITAL,"83825 HIGHWAY 9, ASHLAND, AL, 36251",33.277653,-85.823985,"33.2776532, -85.82398529999999"
52,WHITFIELD REGIONAL HOSPITAL,"105 HIGHWAY 80 EAST, DEMOPOLIS, AL, 36732",32.504829,-87.836573,"32.5048293, -87.836573"
57,LAKELAND COMMUNITY HOSPITAL,"42024 HIGHWAY 195 E, HALEYVILLE, AL, 35565",34.241985,-87.591404,"34.241985, -87.5914038"
70,RUSSELLVILLE HOSPITAL,"15155 HIGHWAY 43, RUSSELLVILLE, AL, 35653",34.510932,-87.718440,"34.510932, -87.7184396"
...,...,...,...,...,...
3295,JENKINS COMMUNITY HOSPITAL,"9480 HIGHWAY 805, JENKINS, KY, 41537",37.171310,-82.634136,"37.17130969999999, -82.63413609999999"
3302,RICHWOOD AREA COMMUNITY HOSPITAL,"RIVERSIDE ADDITION, RICHWOOD, WV, 26261",38.218659,-80.544028,"38.218659, -80.5440284"
3306,OWYHEE COMMUNITY HEALTH FACILITY,", OWYHEE, NV, 89832",41.966584,-116.156588,"41.96658360000001, -116.1565875"
3309,FT YUMA PHS INDIAN HOSP,", YUMA, AZ, 85366",32.687169,-114.624698,"32.687169, -114.6246981"


In [49]:
missing_coords['Coordinates'].isna().sum()

np.int64(0)

In [50]:
hospitals_master_geo_full.columns

Index(['CCN', 'Facility Name', 'Address', 'City', 'State', 'Zip', 'County',
       'Emergency Services', 'Hospital Ownership', 'Closed', 'RUCA',
       'Medicare Payment', 'Number of Beds', 'Closure Date', 'Full Address',
       'Tract_Code', 'Coordinates', 'Prior Name'],
      dtype='object')

Merge coordinates in missing_coords back into main DataFrame:

In [51]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(missing_coords, how='outer', on=['Facility Name','Full Address'])

# Coalesce the coordinates column using combine_first
hospitals_master_geo_full['Coordinates'] = hospitals_master_geo_full['Coordinates_x'].combine_first(hospitals_master_geo_full['Coordinates_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['Coordinates_x', 'Coordinates_y'])


In [52]:
hospitals_master_geo_full.head(2)

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,RUCA,Medicare Payment,Number of Beds,Closure Date,Full Address,Tract_Code,Prior Name,Latitude,Longitude,Coordinates
0,190034,ABBEVILLE GENERAL HOSPITAL,118 N HOSPITAL DR,ABBEVILLE,LA,70510,VERMILION,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaN,NaT,"118 N HOSPITAL DR, ABBEVILLE, LA, 70510",950901.0,,NaN,NaN,"-92.106921121697,29.972129572344"
1,240057,ABBOTT NORTHWESTERN HOSPITAL,800 EAST 28TH STREET,MINNEAPOLIS,MN,55407,HENNEPIN,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaN,NaT,"800 EAST 28TH STREET, MINNEAPOLIS, MN, 55407",125800.0,,NaN,NaN,"-93.262528292723,44.951998305917"


In [53]:
hospitals_master_geo_full.shape

(3304, 20)

In [54]:
hospitals_master_geo_full['Coordinates'].isna().sum()

np.int64(0)

In [55]:
# Fixing a couple of facilities that were incorrectly geocoded
hospitals_master_geo_full.loc[2733,'Coordinates'] = '40.202487340639685, -74.9249632792754'
hospitals_master_geo_full.loc[180,'Coordinates'] = '18.39375056251106, -66.0747418288419'
hospitals_master_geo_full.loc[1210,'Coordinates'] = '18.08347850891205, -67.03997125347175'
hospitals_master_geo_full.loc[656,'Coordinates'] = '15.210976029336237, 145.7241790273805'


Adding latitude and longitude columns since we now have coordinates for every facility.

In [56]:
hospitals_master_geo_full['Latitude'] = hospitals_master_geo_full['Coordinates'].apply(lambda x: x.split(',')[0])
hospitals_master_geo_full['Longitude'] = hospitals_master_geo_full['Coordinates'].apply(lambda x: x.split(',')[1])

I'll also fill in missing RUCA (Rural-Urban Commuting Area code) values, which are a way to designate rural vs urban locations. For that I'll be using a crosswalk downloaded from the USDA.

In [57]:
ruca = pd.read_csv('../data/RUCA-codes-2020-zipcode.csv').rename(columns={'ZIPCode':'Zip','PrimaryRUCA':'RUCA'})[['Zip','RUCA']]

In [58]:
ruca.head(2)

,Zip,RUCA
0,1,10
1,2,10


In [59]:
hospitals_master_geo_full.columns

Index(['CCN', 'Facility Name', 'Address', 'City', 'State', 'Zip', 'County',
       'Emergency Services', 'Hospital Ownership', 'Closed', 'RUCA',
       'Medicare Payment', 'Number of Beds', 'Closure Date', 'Full Address',
       'Tract_Code', 'Prior Name', 'Latitude', 'Longitude', 'Coordinates'],
      dtype='object')

In [60]:
# Merge in RUCA scores
hospitals_master_geo_full = hospitals_master_geo_full.merge(ruca, on='Zip', how='left')

# Coalesce the RUCA column using combine_first
hospitals_master_geo_full['RUCA'] = hospitals_master_geo_full['RUCA_x'].combine_first(hospitals_master_geo_full['RUCA_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['RUCA_x', 'RUCA_y'])

In [61]:
hospitals_master_geo_full['RUCA'].isna().sum()

np.int64(0)

In [62]:
hospitals_master_geo_full.head(2)

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,Medicare Payment,Number of Beds,Closure Date,Full Address,Tract_Code,Prior Name,Latitude,Longitude,Coordinates,RUCA
0,190034,ABBEVILLE GENERAL HOSPITAL,118 N HOSPITAL DR,ABBEVILLE,LA,70510,VERMILION,Yes,Government - Hospital District or Authority,0,NaN,NaN,NaT,"118 N HOSPITAL DR, ABBEVILLE, LA, 70510",950901.0,,-92.106921121697,29.972129572344,"-92.106921121697,29.972129572344",4.0
1,240057,ABBOTT NORTHWESTERN HOSPITAL,800 EAST 28TH STREET,MINNEAPOLIS,MN,55407,HENNEPIN,Yes,Voluntary non-profit - Private,0,NaN,NaN,NaT,"800 EAST 28TH STREET, MINNEAPOLIS, MN, 55407",125800.0,,-93.262528292723,44.951998305917,"-93.262528292723,44.951998305917",1.0


Filling in missing census tracts:

In [63]:
missing_tracts = hospitals_master_geo_full[hospitals_master_geo_full['Tract_Code'].isna()][['Facility Name','Full Address','Coordinates','Latitude','Longitude','Tract_Code']]
                                                                                            

In [64]:
missing_tracts.head(2)

,Facility Name,Full Address,Coordinates,Latitude,Longitude,Tract_Code
5,ABRAZO WEST CAMPUS,"SALLIE, GOODYEAR, AZ, 85395","33.4619617, -112.3528494",33.4619617,-112.3528494,NaN
14,ADVANCED DALLAS HOSPITALS AND CLINICS,"7502 GREENVILLE AVENUE, DALLAS, TX, 75231","32.8837528, -96.7579301",32.8837528,-96.7579301,NaN


In [65]:
# Load coordinates as a GeoDataFrame
gdf_points = gpd.GeoDataFrame(
    missing_tracts, 
    crs='EPSG:4326', 
    geometry=gpd.points_from_xy(missing_tracts['Longitude'], missing_tracts['Latitude'])
)


In [66]:
# Census tract data comes from IPUMS 
census_tracts = gpd.read_file('../data/nhgis0001_shapefile_tl2024_us_tract_2024/US_tract_2024.shp') 

In [67]:
# Ensure both dataframes use the same Coordinate Reference System (CRS)
gdf_points = gdf_points.to_crs(census_tracts.crs)

In [68]:
# Perform a spatial join (place each Points within a Tract)
merged = gpd.sjoin(gdf_points, census_tracts, how='left', predicate='within')
merged['Tract_Code'] = merged['GEOID'].str[-6:]
merged = merged[['Facility Name','Full Address','Coordinates','Latitude','Longitude','Tract_Code']]

In [69]:
merged['Tract_Code'].isna().sum()

np.int64(8)

In [70]:
externalus = merged[merged['Tract_Code'].isna()]
externalus

,Facility Name,Full Address,Coordinates,Latitude,Longitude,Tract_Code
656,COMMONWEALTH HEALTH CENTER,"1 LOWER NAVY HILL ROAD (CK), GARAPAN, MP, 96950","15.210976029336237, 145.7241790273805",15.210976029336237,145.7241790273805,NaN
1000,GOV JUAN F LUIS HOSPITAL & MEDICAL CTR,"#4007 EST DIAMOND RUBY, ST CROIX, VI, 820","17.7378211, -64.7556187",17.7378211,-64.7556187,NaN
1032,GUAM MEMORIAL HOSPITAL AUTHORITY,"85O GOV CARLOS G CAMACHO ROAD, TAMUNING, GU, 9...","13.494396, 144.7759353",13.494396,144.7759353,NaN
1033,GUAM REGIONAL MEDICAL CITY,"133 ROUTE 3, DEDEDO, GU, 96929","13.5256139, 144.8226633",13.5256139,144.8226633,NaN
1484,LBJ TROPICAL MEDICAL CENTER,"FAGAALU VILLAGE, PAGO PAGO, AS, 96799","-14.2901727, -170.6827558",-14.2901727,-170.6827558,NaN
1971,"NORTH HAWAII COMMUNITY HOSPITAL, INC","67 1125 MAMALAHOA HIGHWAY, KAMUELA, HI, 96743","20.0219431, -155.664355",20.0219431,-155.664355,NaN
2390,"ROY LESTER SCHNEIDER HOSPITAL,THE","9048 SUGAR ESTATE, ST THOMAS, VI, 801","18.3408976, -64.9147613",18.3408976,-64.9147613,NaN
3257,WILCOX MEMORIAL HOSPITAL,"3-3420 KUHIO HIGHWAY, LIHUE, HI, 96766","21.9858762, -159.3651263",21.9858762,-159.3651263,NaN


Looks like the facilities outside the contiguous US couldn't be merged with the census shapefile. Reading in those census tract codes separately:

In [71]:
# Load coordinates as a GeoDataFrame
gdf_points = gpd.GeoDataFrame(
    externalus, 
    crs='EPSG:4326', 
    geometry=gpd.points_from_xy(externalus['Longitude'], externalus['Latitude'])
)


In [72]:
# This census tract data comes from US census tigerline shapefiles (https://www.census.gov/cgi-bin/geo/shapefiles/index.php?year=2025&layergroup=Census+Tracts)

# Gather all shapefile paths 
shapefile_folder = Path('../data/Ext_US/')
shapefile_paths = list(shapefile_folder.glob('*/*.shp'))

# Read all files into a list using a list comprehension
gdf_list = [gpd.read_file(path) for path in shapefile_paths]

# Concatenate them and preserve the CRS of the first shapefile
census_tracts = pd.concat(gdf_list, ignore_index=True)
census_tracts.crs = gdf_list[0].crs

In [73]:
# Ensure both dataframes use the same Coordinate Reference System (CRS)
gdf_points = gdf_points.to_crs(census_tracts.crs)

In [74]:
# Perform a spatial join (place each Points within a Tract)
merged_extus = gpd.sjoin(gdf_points, census_tracts, how='left', predicate='within')
merged_extus['Tract_Code'] = merged_extus['GEOID'].str[-6:]
merged_extus = merged_extus[['Facility Name','Full Address','Coordinates','Latitude','Longitude','Tract_Code']]

In [75]:
merged_extus

,Facility Name,Full Address,Coordinates,Latitude,Longitude,Tract_Code
656,COMMONWEALTH HEALTH CENTER,"1 LOWER NAVY HILL ROAD (CK), GARAPAN, MP, 96950","15.210976029336237, 145.7241790273805",15.210976029336237,145.7241790273805,000500
1000,GOV JUAN F LUIS HOSPITAL & MEDICAL CTR,"#4007 EST DIAMOND RUBY, ST CROIX, VI, 820","17.7378211, -64.7556187",17.7378211,-64.7556187,970500
1032,GUAM MEMORIAL HOSPITAL AUTHORITY,"85O GOV CARLOS G CAMACHO ROAD, TAMUNING, GU, 9...","13.494396, 144.7759353",13.494396,144.7759353,955900
1033,GUAM REGIONAL MEDICAL CITY,"133 ROUTE 3, DEDEDO, GU, 96929","13.5256139, 144.8226633",13.5256139,144.8226633,950300
1484,LBJ TROPICAL MEDICAL CENTER,"FAGAALU VILLAGE, PAGO PAGO, AS, 96799","-14.2901727, -170.6827558",-14.2901727,-170.6827558,950700
1971,"NORTH HAWAII COMMUNITY HOSPITAL, INC","67 1125 MAMALAHOA HIGHWAY, KAMUELA, HI, 96743","20.0219431, -155.664355",20.0219431,-155.664355,NaN
2390,"ROY LESTER SCHNEIDER HOSPITAL,THE","9048 SUGAR ESTATE, ST THOMAS, VI, 801","18.3408976, -64.9147613",18.3408976,-64.9147613,961100
3257,WILCOX MEMORIAL HOSPITAL,"3-3420 KUHIO HIGHWAY, LIHUE, HI, 96766","21.9858762, -159.3651263",21.9858762,-159.3651263,NaN


In [76]:
merged_extus['Tract_Code'].isna().sum()

np.int64(2)

Merging externalus back into 'merged', then merging 'merged' back into hospitals_master_geo_full:

In [77]:
merged = merged.merge(merged_extus[['Facility Name','Full Address','Tract_Code']], how='outer', on=['Facility Name','Full Address'])

# Coalesce the tract code column using combine_first
merged['Tract_Code'] = merged['Tract_Code_x'].combine_first(merged['Tract_Code_y'])

# Clean up unneeded columns and drop duplicated rows
merged = merged.drop(columns=['Tract_Code_x', 'Tract_Code_y'])

In [78]:
merged['Tract_Code'].isna().sum()

np.int64(2)

In [79]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(merged[['Facility Name','Full Address','Tract_Code']], how='outer', on=['Facility Name','Full Address'])

# Coalesce the tract code column using combine_first
hospitals_master_geo_full['Tract_Code'] = hospitals_master_geo_full['Tract_Code_x'].combine_first(hospitals_master_geo_full['Tract_Code_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['Tract_Code_x', 'Tract_Code_y'])

In [80]:
hospitals_master_geo_full['Tract_Code'].isna().sum()

np.int64(2)

In [81]:
hospitals_master_geo_full.shape

(3304, 20)

Now that we have census tracts for each hospital, we can narrow down the entire dataset to just rural hospitals based on the 
Federal Office of Rural Health Policy's (FORHP) definition of a rural area. I'm utilizing their dataset here as a crosswalk: https://www.hrsa.gov/rural-health/about-us/what-is-rural/data-files


In [82]:
rural = pd.read_excel('../data/rural-health-areas-data-set.xlsx',sheet_name='Census_Tract_Eligibility').rename(columns={'Tract_2020':'Tract_Code','County_Name_2023':'County'})[['State','County','Tract_Code','FORHP_Rural']]
rural['County'] = rural['County'].str.rsplit(n=1).str[0].str.upper()
rural['County'] = rural['County'].str.replace(' ', '')
rural['County'] = rural['County'].str.replace('\'', '')
hospitals_master_geo_full['County'] = hospitals_master_geo_full['County'].str.replace(' ','')

In [83]:
rural.head(2)

,State,County,Tract_Code,FORHP_Rural
0,AL,AUTAUGA,20100.0,No
1,AL,AUTAUGA,20200.0,No


In [84]:
rural['County'].unique()

array(['AUTAUGA', 'BALDWIN', 'BARBOUR', ..., 'YAUCO', 'ST.JOHN',
       'ST.THOMAS'], dtype=object)

In [85]:
rural.dtypes

State           object
County          object
Tract_Code     float64
FORHP_Rural     object
dtype: object

In [86]:
hospitals_master_geo_full['Tract_Code'] = hospitals_master_geo_full['Tract_Code'].astype('float64')

In [87]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(rural, how='left', on=['State','County','Tract_Code'])

In [88]:
hospitals_master_geo_full.head()[['Facility Name','Tract_Code','FORHP_Rural']]

,Facility Name,Tract_Code,FORHP_Rural
0,ABBEVILLE GENERAL HOSPITAL,950901.0,No
1,ABBOTT NORTHWESTERN HOSPITAL,125800.0,No
2,ABRAZO ARROWHEAD HOSPITAL,615900.0,No
3,ABRAZO CENTRAL CAMPUS,106802.0,No
4,ABRAZO SCOTTSDALE CAMPUS,103303.0,No


We are narrowing down to just rural hospitals for the scope of this project using the FORHP_Rural flag.

In [89]:
hospitals_master_geo_full[(hospitals_master_geo_full['FORHP_Rural'].isna()) & (hospitals_master_geo_full['Closed']==0)][['State','County','Tract_Code','Closed','FORHP_Rural']]


,State,County,Tract_Code,Closed,FORHP_Rural
156,TX,HAYS,31100.0,0,NaN
275,FL,ESCAMBIA,16500.0,0,NaN
294,MO,ST.LOUISCITY,112400.0,0,NaN
298,AK,JUNEAU,400.0,0,NaN
303,LA,E.BATONROUGE,3804.0,0,NaN
...,...,...,...,...,...
3271,VA,WINCHESTERCITY,201.0,0,NaN
3272,CT,WINDHAM,800300.0,0,NaN
3281,LA,E.BATONROUGE,4509.0,0,NaN
3296,CT,NEWHAVEN,140300.0,0,NaN


Looks like some of our data has no value for FORHP_Rural. All hospitals with "Closed" > 0 are definitely rural since they came from the Sheps dataset. There are 160 hospitals that did not close (Closed = 0) that have no value for FORHP_Rural; we will drop those from the dataset. This represents 4.8% of our data.

In [90]:
160/3318

0.04822182037371911

In [91]:
hospitals_master_geo_full = hospitals_master_geo_full[~((hospitals_master_geo_full['FORHP_Rural'].isna()) & (hospitals_master_geo_full['Closed']==0))]



In [92]:
hospitals_master_geo_full = hospitals_master_geo_full[(hospitals_master_geo_full['Closed'] != 0) | (hospitals_master_geo_full['FORHP_Rural'] == 'No')]


In [93]:
hospitals_master_geo_full.shape

(2281, 21)

In [94]:
hospitals_master_geo_full['Closed'].value_counts()

Closed
0    2087
2     107
1      87
Name: count, dtype: int64

Next, pulling in CBSA information for each hospital.
FIPS-CBSA crosswalk comes from here: https://www.nber.org/research/data/census-core-based-statistical-area-cbsa-federal-information-processing-series-fips-county-crosswalk

In [95]:
fips_cbsa_crosswalk = pd.read_csv('../data/cbsa2fipsxw_2023.csv').rename(columns={'countycountyequivalent':'County','state_abbreviation':'State'})
fips_cbsa_crosswalk['County'] = fips_cbsa_crosswalk['County'].str.rsplit(n=1).str[0].str.upper()
fips_cbsa_crosswalk = fips_cbsa_crosswalk.drop(columns=['metropolitandivisioncode','csacode','metropolitandivisiontitle','csatitle','statename','centraloutlyingcounty','fipscountycode','county_in_cbsa'])


Pad county_fips values with zeroes on the left if they are not 5 digits (and convert to string type):

In [96]:
fips_cbsa_crosswalk['county_fips'] = fips_cbsa_crosswalk['county_fips'].astype(str).str.zfill(5)

In [97]:
fips_cbsa_crosswalk.head(2)

,cbsacode,cbsatitle,metropolitanmicropolitanstatis,County,State,fipsstatecode,county_fips
0,33860.0,"Montgomery, AL",Metropolitan Statistical Area,AUTAUGA,AL,1.0,01001
1,19300.0,"Daphne-Fairhope-Foley, AL",Metropolitan Statistical Area,BALDWIN,AL,1.0,01003


In [98]:
fips_cbsa_crosswalk['County'] = fips_cbsa_crosswalk['County'].str.replace(' ', '')
fips_cbsa_crosswalk['County'] = fips_cbsa_crosswalk['County'].str.replace('\'', '')

In [99]:
fips_cbsa_crosswalk.dtypes

cbsacode                          float64
cbsatitle                          object
metropolitanmicropolitanstatis     object
County                             object
State                              object
fipsstatecode                     float64
county_fips                        object
dtype: object

In [100]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(fips_cbsa_crosswalk, how='left', on=['County','State'])

In [101]:
hospitals_master_geo_full[hospitals_master_geo_full['county_fips'].isna()]

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,...,Longitude,Coordinates,RUCA,Tract_Code,FORHP_Rural,cbsacode,cbsatitle,metropolitanmicropolitanstatis,fipsstatecode,county_fips
528,NaN,DOCTORS HOSPITAL AT DEER CREEK,815 SOUTH 10TH STREET,LEESVILLE,LA,71446,VERNONPARISH,NaN,NaN,2,...,-93.2688034,"31.1357678, -93.2688034",7.0,950501.0,NaN,NaN,NaN,NaN,NaN,NaN
532,NaN,DOCTORS HOSPITAL OF OPELOUSAS,3983 I 49 SOUTH SERVICE ROAD,OPELOUSAS,LA,70570,ST.LANDRYPARISH,NaN,NaN,1,...,-92.0795016,"30.4717797, -92.0795016",2.0,961701.0,NaN,NaN,NaN,NaN,NaN,NaN
1226,NaN,MERCYONE PRIMGHAR MEDICAL CENTER,255 N WELCH AVENUE,PRIMGHAR,IA,51245,O'BRIEN,NaN,NaN,2,...,43.088836290895,"-95.622895045875,43.088836290895",10.0,490200.0,NaN,NaN,NaN,NaN,NaN,NaN
1768,NaN,SITKA COMMUNITY HOSPITAL,209 MOLLER DRIVE,SITKA,AK,99835,"SITKA,",NaN,NaN,1,...,57.058099272476,"-135.346159635252,57.058099272476",7.0,200.0,NaN,NaN,NaN,NaN,NaN,NaN
1788,NaN,SOUTHEAST REGIONAL MEDICAL CENTER,719 AVENUE G,KENTWOOD,LA,70444,TANGIPAHOAPARISH,NaN,NaN,1,...,30.936823415518,"-90.517113361597,30.936823415518",10.0,953300.0,NaN,NaN,NaN,NaN,NaN,NaN


Adding county FIPS information manually for rows that don't have it:

In [102]:
hospitals_master_geo_full.loc[528,'county_fips'] = '22115'
hospitals_master_geo_full.loc[532,'county_fips'] = '22097'
hospitals_master_geo_full.loc[1226,'county_fips'] = '19141'
hospitals_master_geo_full.loc[1768,'county_fips'] = '02220'
hospitals_master_geo_full.loc[1788,'county_fips'] = '22105'


In [103]:
hospitals_master_geo_full["metropolitanmicropolitanstatis"] = hospitals_master_geo_full["metropolitanmicropolitanstatis"].fillna("Neither")

In [104]:
hospitals_master_geo_full = hospitals_master_geo_full.rename(columns={'cbsacode':'CBSA_Code','cbsatitle':'CBSA_Title','metropolitanmicropolitanstatis':'Metro_Status','fipsstatecode':'State_FIPS','county_fips':'County_FIPS'})


In [105]:
hospitals_master_geo_full.columns

Index(['CCN', 'Facility Name', 'Address', 'City', 'State', 'Zip', 'County',
       'Emergency Services', 'Hospital Ownership', 'Closed',
       'Medicare Payment', 'Number of Beds', 'Closure Date', 'Full Address',
       'Prior Name', 'Latitude', 'Longitude', 'Coordinates', 'RUCA',
       'Tract_Code', 'FORHP_Rural', 'CBSA_Code', 'CBSA_Title', 'Metro_Status',
       'State_FIPS', 'County_FIPS'],
      dtype='object')

Fill in missing state FIPS values:

In [106]:
state_fips_crosswalk = fips_cbsa_crosswalk[fips_cbsa_crosswalk['fipsstatecode'].notna()][['State','fipsstatecode']].drop_duplicates().reset_index(drop=True).rename(columns={'fipsstatecode':'State_FIPS'})


In [107]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(state_fips_crosswalk, how='left', on='State')

# Coalesce the state FIPS code column using combine_first
hospitals_master_geo_full['State_FIPS'] = hospitals_master_geo_full['State_FIPS_x'].combine_first(hospitals_master_geo_full['State_FIPS_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['State_FIPS_x', 'State_FIPS_y'])

Manually adding State FIPS values for Guam and Mariana Islands:

In [108]:
hospitals_master_geo_full.loc[[290,291],'State_FIPS'] = 66
hospitals_master_geo_full.loc[179,'State_FIPS'] = 69

Getting missing CCNs:

In [109]:
hospitals_master_geo_full[hospitals_master_geo_full['CCN'].isna()]

,CCN,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,...,Longitude,Coordinates,RUCA,Tract_Code,FORHP_Rural,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS
6,NaN,ACOMA-CANONCITO-LAGUNA SERVICE UNIT,80B VETERANS BLVD,ACOMA,NM,87034,CIBOLA,NaN,NaN,1,...,35.069218425753,"-107.577092689666,35.069218425753",2.0,941500.0,Yes,NaN,NaN,Neither,35006,35.0
43,NaN,ADVENTIST HEALTH FEATHER RIVER,5974 PENTZ ROAD,PARADISE,CA,95969,BUTTE,NaN,NaN,2,...,39.755682251776,"-121.572237161706,39.755682251776",4.1,2000.0,No,17020.0,"Chico, CA",Metropolitan Statistical Area,06007,6.0
76,NaN,ALBANY AREA HOSPITAL,300 3RD AVENUE,ALBANY,MN,56307,STEARNS,NaN,NaN,2,...,45.63915451062,"-94.567073824853,45.63915451062",2.0,10402.0,No,41060.0,"St. Cloud, MN",Metropolitan Statistical Area,27145,27.0
78,NaN,ALBERT LEA - MAYO CLINIC HEALTH SYSTEM,404 WEST FOUNTAIN STREET,ALBERT LEA,MN,56007,FREEBORN,NaN,NaN,1,...,43.651590633716,"-93.372490760223,43.651590633716",4.0,180400.0,Yes,10660.0,"Albert Lea, MN",Micropolitan Statistical Area,27047,27.0
79,NaN,ALBERT LINDLEY LEE MEMORIAL HOSPITAL,510 SOUTH 4TH STREET,FULTON,NY,13069,OSWEGO,NaN,NaN,1,...,43.314646213714,"-76.405705022172,43.314646213714",4.1,21102.0,Yes,45060.0,"Syracuse, NY",Metropolitan Statistical Area,36075,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,NaN,WEIMAR MEDICAL CENTER,400 YOUENS DR,WEIMAR,TX,78962,COLORADO,NaN,NaN,2,...,29.693310210662,"-96.787333171798,29.693310210662",10.1,750300.0,Yes,NaN,NaN,Neither,48089,48.0
2283,NaN,WISE REGIONAL HEALTH SYSTEM-BRIDGEPORT,1901 DOCTORS HOSPITAL DRIVE,BRIDGEPORT,TX,76426,WISE,NaN,NaN,1,...,33.221794466981,"-97.761881722791,33.221794466981",7.0,150500.0,Yes,19100.0,"Dallas-Fort Worth-Arlington, TX",Metropolitan Statistical Area,48497,48.0
2284,NaN,WM B KESSLER MEMORIAL HOSPITAL,600 SO WHITE HORSE PIKE,HAMMONTON,NJ,8037,ATLANTIC,NaN,NaN,1,...,39.630573063288,"-74.771056217374,39.630573063288",4.1,10900.0,Yes,12100.0,"Atlantic City-Hammonton, NJ",Metropolitan Statistical Area,34001,34.0
2288,NaN,WOODLAND MEDICAL CTR,1910 CHEROKEE AVENUE SW,CULLMAN,AL,35055,CULLMAN,NaN,NaN,2,...,34.151616163828,"-86.843018009046,34.151616163828",4.0,965002.0,Yes,18980.0,"Cullman, AL",Micropolitan Statistical Area,01043,1.0


In [110]:
hospitals_master_geo_full.loc[6,'CCN'] = '320070'
hospitals_master_geo_full.loc[6,'Hospital Ownership'] = 'Governmental, Federal'
hospitals_master_geo_full.loc[6,'Emergency Services'] = 'Yes'

In [111]:
hospitals_master_geo_full.loc[43,'CCN'] = '050225'
hospitals_master_geo_full.loc[43,'Hospital Ownership'] = 'Voluntary Nonprofit, Church'
hospitals_master_geo_full.loc[43,'Emergency Services'] = 'Yes'

In [112]:
missing_ccn = hospitals_master_geo_full[hospitals_master_geo_full['CCN'].isna()]
missing_ccn.to_csv('../data/missing_ccn.txt')

In [113]:
_2019_hospitals = pd.read_csv('../data/Hospital General Information_2019.csv')

In [114]:
import pandas as pd
from rapidfuzz import process, fuzz

def load_data():
    # Load your existing file with 190 hospitals
    # Replace with your actual file path (works with .csv or .xlsx)
    my_hospitals = pd.read_csv('../data/missing_ccn.txt')
    
    # Load the historical CMS reference file containing the CCNs
    # Replace with your downloaded CMS file path
    cms_reference = _2019_hospitals
    
    return my_hospitals, cms_reference

def clean_zip(zip_code):
    """Standardizes ZIP codes to a 5-digit string format."""
    if pd.isna(zip_code):
        return ""
    # Convert float/int to string and strip any decimal or '99999-XXXX' trailing digits
    zip_str = str(zip_code).split('.')[0].split('-')[0].strip()
    return zip_str.zfill(5) # Pads with leading zeros if necessary (e.g., '02108')

def find_ccn_matches(my_df, ref_df, score_cutoff=80):
    """
    Matches hospitals based on State, Fuzzy Name matching, and ZIP Code verification.
    """
    # Standardize names and locations for uniform matching
    my_df['clean_name'] = my_df['Facility Name'].astype(str).str.strip().str.upper()
    my_df['clean_state'] = my_df['State'].astype(str).str.strip().str.upper()
    my_df['clean_zip'] = my_df['Zip'].apply(clean_zip)
    
    # Adjust reference column names based on your CMS reference file schema
    ref_df['clean_name'] = ref_df['Facility Name'].astype(str).str.strip().str.upper()
    ref_df['clean_state'] = ref_df['State'].astype(str).str.strip().str.upper()
    ref_df['clean_zip'] = ref_df['ZIP Code'].apply(clean_zip) 
    
    # Create empty columns for the results
    my_df['Matched_CCN'] = None
    my_df['Match_Score'] = 0.0
    my_df['Matched_Ref_Name'] = None
    my_df['Zip_Verified'] = "No Match"

    # Iterate through rows to find the best match within the same state
    for idx, row in my_df.iterrows():
        state = row['clean_state']
        name = row['clean_name']
        user_zip = row['clean_zip']
        
        # Filter reference data by state to speed up and limit false positives
        state_ref = ref_df[ref_df['clean_state'] == state]
        
        if not state_ref.empty:
            ref_names = state_ref['clean_name'].tolist()
            
            # Find top match using token_sort_ratio
            match = process.extractOne(name, ref_names, scorer=fuzz.token_sort_ratio)
            
            if match:
                matched_name, name_score, ref_idx = match
                actual_ref_row = state_ref.iloc[ref_idx]
                ref_zip = actual_ref_row['clean_zip']
                
                # Apply ZIP code verification adjustment
                if user_zip and ref_zip and (user_zip == ref_zip):
                    # Boost score if zip codes align perfectly
                    final_score = min(name_score + 15, 100) 
                    zip_status = "Exact Match"
                else:
                    # Penalize score if zip codes mismatch
                    final_score = max(name_score - 15, 0)
                    zip_status = "Mismatch / Missing"
                
                # Verify if final adjusted score meets your threshold
                if final_score >= score_cutoff:
                    my_df.at[idx, 'Matched_CCN'] = actual_ref_row['Facility ID'] 
                    my_df.at[idx, 'Match_Score'] = final_score
                    my_df.at[idx, 'Matched_Ref_Name'] = actual_ref_row['Facility Name']
                    my_df.at[idx, 'Zip_Verified'] = zip_status

    # Drop temporary processing columns
    my_df = my_df.drop(columns=['clean_name', 'clean_state', 'clean_zip'])
    return my_df

if __name__ == "__main__":
    try:
        user_df, cms_df = load_data()
        
        print("Starting fuzzy matching with ZIP code verification...")
        matched_output = find_ccn_matches(user_df, cms_df, score_cutoff=80)
        
        matched_output.to_csv('matched_hospitals_with_zip.csv', index=False)
        print("Matching complete! Saved to 'matched_hospitals_with_zip.csv'.")
        
        matches_found = matched_output['Matched_CCN'].notna().sum()
        exact_zips = (matched_output['Zip_Verified'] == 'Exact Match').sum()
        print(f"Successfully matched {matches_found} out of {len(matched_output)} hospitals.")
        print(f"Of those matches, {exact_zips} had perfectly verified ZIP codes.")
        
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your file paths.")


Starting fuzzy matching with ZIP code verification...
Matching complete! Saved to 'matched_hospitals_with_zip.csv'.
Successfully matched 54 out of 192 hospitals.
Of those matches, 53 had perfectly verified ZIP codes.


In [115]:
matched_output = matched_output[['Facility Name','Address','City','State','Matched_CCN']].rename(columns={'Matched_CCN':'CCN'})

In [116]:
matched_output = matched_output[matched_output['CCN']!='None']

In [117]:
hospitals_master_geo_full = hospitals_master_geo_full.merge(matched_output, how='left', on=['Facility Name','Address','City','State'])

In [118]:
# Coalesce the CCN columns using combine_first
hospitals_master_geo_full['CCN'] = hospitals_master_geo_full['CCN_x'].combine_first(hospitals_master_geo_full['CCN_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['CCN_x', 'CCN_y'])

In [119]:
hospitals_master_geo_full['CCN'].isna().sum()

np.int64(138)

In [120]:
missing_ccn = hospitals_master_geo_full[hospitals_master_geo_full['CCN'].isna()]
missing_ccn.to_csv('../data/missing_ccn2.txt')

In [121]:
_2015_hospitals = pd.read_csv('../data/pos2015.csv')

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_96502/2674488354.py:1: DtypeWarning: Columns (0,8,12,13,16,17,22,24,28,49,50,51,52,53,58,59,76,77,78,79,82,88,89,90,91,93,96,100,101,103,104,106,107,110,111,112,113,114,115,130,131,133,134,135,136,137,140,145,148,156,157,158,165,166,167,173,174,175,179,180,181,188,189,191,192,193,200,201,202,209,210,211,213,214,215,222,223,224,225,226,227,229,230,231,239,240,242,243,244,245,246,247,249,250,251,252,253,254,255,256,257,260,261,262,263,264,265,267,268,269,271,272,273,276,277,278,280,281,282,283,284,285,287,288,289,292,293,294,295,296,297,298,299,300,302,303,304,307,308,309,310,311,312,313,452,453,454,455,456,457,458,459,460,461,462,463,464) have mixed types. Specify dtype option on import or set low_memory=False.
  _2015_hospitals = pd.read_csv('../data/pos2015.csv')


In [122]:
import pandas as pd
from rapidfuzz import process, fuzz

def load_data():
    # Load your existing file with 190 hospitals
    # Replace with your actual file path (works with .csv or .xlsx)
    my_hospitals = pd.read_csv('../data/missing_ccn2.txt')
    
    # Load the historical CMS reference file containing the CCNs
    # Replace with your downloaded CMS file path
    cms_reference = _2015_hospitals
    
    return my_hospitals, cms_reference

def clean_zip(zip_code):
    """Standardizes ZIP codes to a 5-digit string format."""
    if pd.isna(zip_code):
        return ""
    # Convert float/int to string and strip any decimal or '99999-XXXX' trailing digits
    zip_str = str(zip_code).split('.')[0].split('-')[0].strip()
    return zip_str.zfill(5) # Pads with leading zeros if necessary (e.g., '02108')

def find_ccn_matches(my_df, ref_df, score_cutoff=80):
    """
    Matches hospitals based on State, Fuzzy Name matching, and ZIP Code verification.
    """
    # Standardize names and locations for uniform matching
    my_df['clean_name'] = my_df['Facility Name'].astype(str).str.strip().str.upper()
    my_df['clean_state'] = my_df['State'].astype(str).str.strip().str.upper()
    my_df['clean_zip'] = my_df['Zip'].apply(clean_zip)
    
    # Adjust reference column names based on your CMS reference file schema
    ref_df['clean_name'] = ref_df['fac_name'].astype(str).str.strip().str.upper()
    ref_df['clean_state'] = ref_df['state_cd'].astype(str).str.strip().str.upper()
    ref_df['clean_zip'] = ref_df['zip_cd'].apply(clean_zip) 
    
    # Create empty columns for the results
    my_df['Matched_CCN'] = None
    my_df['Match_Score'] = 0.0
    my_df['Matched_Ref_Name'] = None
    my_df['Zip_Verified'] = "No Match"

    # Iterate through rows to find the best match within the same state
    for idx, row in my_df.iterrows():
        state = row['clean_state']
        name = row['clean_name']
        user_zip = row['clean_zip']
        
        # Filter reference data by state to speed up and limit false positives
        state_ref = ref_df[ref_df['clean_state'] == state]
        
        if not state_ref.empty:
            ref_names = state_ref['clean_name'].tolist()
            
            # Find top match using token_sort_ratio
            match = process.extractOne(name, ref_names, scorer=fuzz.token_sort_ratio)
            
            if match:
                matched_name, name_score, ref_idx = match
                actual_ref_row = state_ref.iloc[ref_idx]
                ref_zip = actual_ref_row['clean_zip']
                
                # Apply ZIP code verification adjustment
                if user_zip and ref_zip and (user_zip == ref_zip):
                    # Boost score if zip codes align perfectly
                    final_score = min(name_score + 15, 100) 
                    zip_status = "Exact Match"
                else:
                    # Penalize score if zip codes mismatch
                    final_score = max(name_score - 15, 0)
                    zip_status = "Mismatch / Missing"
                
                # Verify if final adjusted score meets your threshold
                if final_score >= score_cutoff:
                    my_df.at[idx, 'Matched_CCN'] = actual_ref_row['prvdr_num'] 
                    my_df.at[idx, 'Match_Score'] = final_score
                    my_df.at[idx, 'Matched_Ref_Name'] = actual_ref_row['fac_name']
                    my_df.at[idx, 'Zip_Verified'] = zip_status

    # Drop temporary processing columns
    my_df = my_df.drop(columns=['clean_name', 'clean_state', 'clean_zip'])
    return my_df

if __name__ == "__main__":
    try:
        user_df, cms_df = load_data()
        
        print("Starting fuzzy matching with ZIP code verification...")
        matched_output = find_ccn_matches(user_df, cms_df, score_cutoff=80)
        
        matched_output.to_csv('matched_hospitals_with_zip.csv', index=False)
        print("Matching complete! Saved to 'matched_hospitals_with_zip.csv'.")
        
        matches_found = matched_output['Matched_CCN'].notna().sum()
        exact_zips = (matched_output['Zip_Verified'] == 'Exact Match').sum()
        print(f"Successfully matched {matches_found} out of {len(matched_output)} hospitals.")
        print(f"Of those matches, {exact_zips} had perfectly verified ZIP codes.")
        
    except FileNotFoundError as e:
        print(f"Error loading files: {e}. Please check your file paths.")


Starting fuzzy matching with ZIP code verification...
Matching complete! Saved to 'matched_hospitals_with_zip.csv'.
Successfully matched 110 out of 138 hospitals.
Of those matches, 109 had perfectly verified ZIP codes.


In [123]:
matched_output = matched_output[['Facility Name','Address','City','State','Matched_CCN']].rename(columns={'Matched_CCN':'CCN'})
matched_output = matched_output[matched_output['CCN']!='None']
hospitals_master_geo_full = hospitals_master_geo_full.merge(matched_output, how='left', on=['Facility Name','Address','City','State'])
# Coalesce the CCN columns using combine_first
hospitals_master_geo_full['CCN'] = hospitals_master_geo_full['CCN_x'].combine_first(hospitals_master_geo_full['CCN_y'])

# Clean up unneeded columns and drop duplicated rows
hospitals_master_geo_full = hospitals_master_geo_full.drop(columns=['CCN_x', 'CCN_y'])

In [124]:
hospitals_master_geo_full['CCN'].isna().sum()

np.int64(31)

In [125]:
hospitals_master_geo_full[hospitals_master_geo_full['CCN'].isna()][['Facility Name','Address','City','State','Zip','County','CCN','Hospital Ownership','Emergency Services']]


,Facility Name,Address,City,State,Zip,County,CCN,Hospital Ownership,Emergency Services
141,ASCENSION ST. VINCENT DUNN,1600 23RD STREET,BEDFORD,IN,47421,LAWRENCE,None,NaN,NaN
158,AUDRAIN COMMUNITY HOSPITAL,620 E MONROE,MEXICO,MO,65265,AUDRAIN,None,NaN,NaN
285,BLESSING HEALTH KEOKUK,1600 MORGAN STREET,KEOKUK,IA,52632,LEE,None,NaN,NaN
327,CALLAWAY COMMUNITY HOSPITAL,10 SOUTH HOSPITAL DRIVE,FULTON,MO,65251,CALLAWAY,None,NaN,NaN
374,CENTRAL HOSPITAL OF BOWIE,705 E GREENWOOD AVE,BOWIE,TX,76230,MONTAGUE,None,NaN,NaN
448,COMMUNITY HEALTHCARE SYSTEM - ST. MARYS,206 GRAND AVE,ST. MARYS,KS,66536,POTTAWATOMIE,None,NaN,NaN
509,DEER PARK HOSPITAL,1015 EAST D STREET,DEER PARK,WA,99006,SPOKANE,None,NaN,NaN
584,EPIC MEDICAL CENTER,1 HOSPITAL DRIVE,EUFAULA,OK,74432,MCINTOSH,None,NaN,NaN
635,FRANCISCAN SKEMP MEDICAL CENTER,464 S ST JOSEPH AVE,ARCADIA,WI,54612,TREMPEALEAU,None,NaN,NaN
646,FT YUMA PHS INDIAN HOSP,,YUMA,AZ,85366,YUMA,NaN,NaN,NaN


In [126]:
hospitals_master_geo_full.loc[141,'CCN'] = '151335'
hospitals_master_geo_full.loc[141,'Hospital Ownership'] = 'Voluntary non-profit - Church'
hospitals_master_geo_full.loc[141,'Emergency Services'] = 'Yes'


In [127]:
hospitals_master_geo_full.loc[158,'CCN'] = '260064'
hospitals_master_geo_full.loc[158,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[158,'Emergency Services'] = 'Yes'

In [128]:
hospitals_master_geo_full.loc[285,'CCN'] = '160008'
hospitals_master_geo_full.loc[285,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[285,'Emergency Services'] = 'Yes'

In [129]:
hospitals_master_geo_full.loc[327,'CCN'] = '260209'
hospitals_master_geo_full.loc[327,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[327,'Emergency Services'] = 'Yes'

In [130]:
hospitals_master_geo_full.loc[374,'CCN'] = '450497'
hospitals_master_geo_full.loc[374,'Hospital Ownership'] = 'Governmental, City'
hospitals_master_geo_full.loc[374,'Emergency Services'] = 'Yes'

In [131]:
hospitals_master_geo_full.loc[448,'CCN'] = '171354'
hospitals_master_geo_full.loc[448,'Hospital Ownership'] = 'Voluntary Nonprofit, Private'
hospitals_master_geo_full.loc[448,'Emergency Services'] = 'Yes'

In [132]:
hospitals_master_geo_full.loc[509,'CCN'] = '501306'
hospitals_master_geo_full.loc[509,'Hospital Ownership'] = 'Voluntary Nonprofit, Private'
hospitals_master_geo_full.loc[509,'Emergency Services'] = 'Yes'

In [133]:
hospitals_master_geo_full.loc[584,'CCN'] = '370169'
hospitals_master_geo_full.loc[584,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[584,'Emergency Services'] = 'Yes'

In [134]:
hospitals_master_geo_full.loc[635,'CCN'] = '520004'
hospitals_master_geo_full.loc[635,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[635,'Emergency Services'] = 'Yes'

FT YUMA PHS INDIAN HOSP is an outpatient facility without a Medicare CCN, so we'll drop it from the dataset. Same with OWYHEE COMMUNITY HEALTH FACILITY.

In [135]:
hospitals_master_geo_full.loc[677,'CCN'] = '451302'
hospitals_master_geo_full.loc[677,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[677,'Emergency Services'] = 'Yes'

In [136]:
hospitals_master_geo_full.loc[922,'CCN'] = '441324'
hospitals_master_geo_full.loc[922,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[922,'Emergency Services'] = 'Yes'

In [137]:
hospitals_master_geo_full.loc[1272,'CCN'] = '343427'
hospitals_master_geo_full.loc[1272,'Hospital Ownership'] = 'Voluntary non-profit - Church'
hospitals_master_geo_full.loc[1272,'Emergency Services'] = 'No'

In [138]:
hospitals_master_geo_full.loc[1313,'CCN'] = '490027'
hospitals_master_geo_full.loc[1313,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[1313,'Emergency Services'] = 'Yes'

In [139]:
hospitals_master_geo_full.loc[1502,'CCN'] = '440064'
hospitals_master_geo_full.loc[1502,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[1502,'Emergency Services'] = 'Yes'

In [140]:
hospitals_master_geo_full.loc[1534,'CCN'] = '360356'
hospitals_master_geo_full.loc[1534,'Hospital Ownership'] = 'Proprietary, Other'
hospitals_master_geo_full.loc[1534,'Emergency Services'] = 'Yes'

In [141]:
hospitals_master_geo_full.loc[1559,'CCN'] = '260004'
hospitals_master_geo_full.loc[1559,'Hospital Ownership'] = 'Governmental, County'
hospitals_master_geo_full.loc[1559,'Emergency Services'] = 'Yes'

In [142]:
hospitals_master_geo_full.loc[1560,'CCN'] = '251332'
hospitals_master_geo_full.loc[1560,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[1560,'Emergency Services'] = 'Yes'

In [143]:
hospitals_master_geo_full.loc[1736,'CCN'] = '370103'
hospitals_master_geo_full.loc[1736,'Hospital Ownership'] = 'Governmental, City'
hospitals_master_geo_full.loc[1736,'Emergency Services'] = 'Yes'

In [144]:
hospitals_master_geo_full.loc[1793,'CCN'] = '200040'
hospitals_master_geo_full.loc[1793,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[1793,'Emergency Services'] = 'Yes'

In [145]:
hospitals_master_geo_full.loc[1931,'CCN'] = '140143'
hospitals_master_geo_full.loc[1931,'Hospital Ownership'] = 'Voluntary Nonprofit, Church'
hospitals_master_geo_full.loc[1931,'Emergency Services'] = 'Yes'

In [146]:
hospitals_master_geo_full.loc[2000,'CCN'] = '670117'
hospitals_master_geo_full.loc[2000,'Hospital Ownership'] = 'Proprietary'
hospitals_master_geo_full.loc[2000,'Emergency Services'] = 'Yes'

In [147]:
hospitals_master_geo_full.loc[2050,'CCN'] = '010174'
hospitals_master_geo_full.loc[2050,'Hospital Ownership'] = 'Proprietary, Partnership'
hospitals_master_geo_full.loc[2050,'Emergency Services'] = 'Yes'

In [148]:
hospitals_master_geo_full.loc[2053,'CCN'] = '281301'
hospitals_master_geo_full.loc[2053,'Emergency Services'] = 'Yes'

In [149]:
hospitals_master_geo_full.loc[2088,'CCN'] = '050325'
hospitals_master_geo_full.loc[2088,'Hospital Ownership'] = 'Government - County'
hospitals_master_geo_full.loc[2088,'Emergency Services'] = 'Yes'

In [150]:
hospitals_master_geo_full.loc[2182,'CCN'] = '390071'
hospitals_master_geo_full.loc[2182,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[2182,'Emergency Services'] = 'Yes'

In [151]:
hospitals_master_geo_full.loc[2190,'CCN'] = '390084'
hospitals_master_geo_full.loc[2190,'Hospital Ownership'] = 'Voluntary Nonprofit, Other'
hospitals_master_geo_full.loc[2190,'Emergency Services'] = 'Yes'

In [152]:
hospitals_master_geo_full.loc[2204,'CCN'] = '370243'
hospitals_master_geo_full.loc[2204,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[2204,'Emergency Services'] = 'Yes'

In [153]:
hospitals_master_geo_full.loc[2235,'CCN'] = '670114'
hospitals_master_geo_full.loc[2235,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[2235,'Emergency Services'] = 'Yes'

In [154]:
hospitals_master_geo_full.loc[2283,'CCN'] = '670052'
hospitals_master_geo_full.loc[2283,'Hospital Ownership'] = 'Proprietary, Corporation'
hospitals_master_geo_full.loc[2283,'Emergency Services'] = 'Yes'

Investigating duplicated CCNs:

In [155]:
hospitals_master_geo_full[hospitals_master_geo_full['CCN'].duplicated(keep=False)][['CCN','Facility Name']]

,CCN,Facility Name
221,260162,BARNES-JEWISH WEST COUNTY HOSPITAL
222,260162,BARNES-JEWISH WEST COUNTY HOSPITAL
411,260180,CHRISTIAN HOSPITAL NORTHEAST
412,260180,CHRISTIAN HOSPITAL NORTHEAST
635,520004,FRANCISCAN SKEMP MEDICAL CENTER
646,NaN,FT YUMA PHS INDIAN HOSP
687,210044,GREATER BALTIMORE MEDICAL CENTER
688,210044,GREATER BALTIMORE MEDICAL CENTER
877,490101,INOVA FAIR OAKS HOSPITAL
878,490101,INOVA FAIR OAKS HOSPITAL


In [156]:
hospitals_master_geo_full.loc[1276:1277,:][['Facility Name','Address','City','State','County','Closure Date','CCN','County_FIPS']]

,Facility Name,Address,City,State,County,Closure Date,CCN,County_FIPS
1276,MISSOURI BAPTIST MEDICAL CENTER,3015 N BALLAS RD,SAINT LOUIS,MO,ST.LOUIS,NaT,260108,29189
1277,MISSOURI BAPTIST MEDICAL CENTER,3015 N BALLAS RD,SAINT LOUIS,MO,ST.LOUIS,NaT,260108,29510


Looks like most of these pairs have one row with an incorrect county FIPS. I'll drop those rows.

In [157]:
hospitals_master_geo_full = hospitals_master_geo_full.drop([222, 412, 635, 688, 877, 880, 883, 1155, 1208, 1212, 1277, 1390, 1632, 1811, 1818, 1894, 1898, 2119, 2164])



In [158]:
hospitals_master_geo_full['CCN'].isna().sum()

np.int64(2)

In [159]:
hospitals_master_geo_full = hospitals_master_geo_full[~hospitals_master_geo_full['CCN'].isna()]

In [160]:
hospitals_master_geo_full[hospitals_master_geo_full['CCN'].duplicated(keep=False)][['CCN','Facility Name']]

,CCN,Facility Name


Now that we've confirmed each CCN is unique, we can use this as the primary merge key going forward.

Also bringing in NPI numbers via this crosswalk: https://www.nber.org/research/data/national-provider-identifier-npi-medicare-ccn-crosswalk

In [161]:
npi_xw = pd.read_csv('../data/npi_medicarexw.csv')[['npi','othpid']].rename(columns={'npi':'NPI', 'othpid':'CCN'})


In [162]:
hospitals_npi = hospitals_master_geo_full[['Facility Name','CCN']].merge(npi_xw, how='left', on='CCN')

In [163]:
hospitals_master_geo_full[hospitals_master_geo_full[['Facility Name','CCN']].duplicated(keep=False)]

,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,Medicare Payment,...,Coordinates,RUCA,Tract_Code,FORHP_Rural,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS,CCN


In [164]:
# Dropping duplicated facilities 
hospitals_master_geo_full = hospitals_master_geo_full.drop([737,118])

In [165]:
hospitals_npi.shape

(5304, 3)

In [166]:
hospitals_npi.isna().sum()

Facility Name      0
CCN                0
NPI              304
dtype: int64

In [167]:
hospitals_master_geo_full.shape

(2276, 26)

In [168]:
hospitals_master_geo_full.isna().sum()

Facility Name            0
Address                  0
City                     0
State                    0
Zip                      0
County                   0
Emergency Services     161
Hospital Ownership     162
Closed                   0
Medicare Payment      2085
Number of Beds        2085
Closure Date          2088
Full Address             0
Prior Name               0
Latitude                 0
Longitude                0
Coordinates              0
RUCA                     0
Tract_Code               0
FORHP_Rural              6
CBSA_Code               89
CBSA_Title              89
Metro_Status             0
County_FIPS              0
State_FIPS               0
CCN                      0
dtype: int64

Converting CCN to 6-digit string:

In [169]:
hospitals_master_geo_full['CCN'] = hospitals_master_geo_full['CCN'].astype(str)

In [170]:
hospitals_master_geo_full['CCN'] = hospitals_master_geo_full['CCN'].str.zfill(6)

Filling in missing ownership types with most common ownership type:

In [171]:
# Exporting master hospital table to csv for use in other notebooks
hospitals_master_geo_full.drop(columns=['FORHP_Rural','Medicare Payment','Number of Beds','Coordinates','Latitude','Longitude','Address','City']).to_csv('../data/hospitals_master.csv',index=False)


In [172]:
hospitals_master_geo_full['Closed'].value_counts()

Closed
0    2085
2     106
1      85
Name: count, dtype: int64

In [173]:
hospitals_master_geo_full[hospitals_master_geo_full['CCN'].duplicated(keep=False)]

,Facility Name,Address,City,State,Zip,County,Emergency Services,Hospital Ownership,Closed,Medicare Payment,...,Coordinates,RUCA,Tract_Code,FORHP_Rural,CBSA_Code,CBSA_Title,Metro_Status,County_FIPS,State_FIPS,CCN
